# Training-data scaling (Figure 4C)

Top-1 classification accuracy of the `cell_dino` attention model (gene-KO classifier, C = 1,001) as a function of the per-set cell count, for four **training-set sizes** (1.5M → 50M cells). Larger training pools lift the whole accuracy curve; the x-axis is log-scaled so the early-bin gains (10 → 100 cells) stay visible.

Inputs are the per-training-size evaluation JSONs produced by the attention pipeline, committed alongside this notebook:

| file | training set |
| --- | --- |
| `attn_train1p5M.json` | 1.5M cells |
| `attn_train5M.json` | 5M cells |
| `attn_train15M.json` | 15M cells |
| `attn_train50M.json` | 50M cells |


## Imports

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, MultipleLocator, NullFormatter

# Keep text editable in Illustrator (SVG keeps <text> elements; PDF uses TrueType).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

FIGURES_DIR = Path("../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data paths

Per-training-size evaluation JSONs, committed alongside this notebook, ordered smallest → largest pool.

In [ ]:
FIGURE_DATA = Path(".")  # JSONs sit alongside this notebook

RUNS = [
    ("attn_train1p5M.json", "1.5M"),
    ("attn_train5M.json", "5M"),
    ("attn_train15M.json", "15M"),
    ("attn_train50M.json", "50M cells"),
]

## Configuration

Top-1 accuracy on a log x-axis, one curve per training-set size coloured by `viridis` (dark = largest pool).

In [ ]:
ACC_KEY = "mean_accuracy_top1"   # "mean_accuracy_top1" or "mean_accuracy_top5"

# viridis, ordered dark -> light for largest -> smallest training pool
COLORS = cm.viridis(np.linspace(0.9, 0.08, len(RUNS)))

## Load

Each JSON carries `n_cells_list` and the mean accuracy per bin (mean over the C classes).

In [ ]:
def load(path: Path):
    d = json.loads(Path(path).read_text())
    df = pd.DataFrame({"n_cells": d["n_cells_list"], "acc": d[ACC_KEY]})
    return df, int(d["n_classes"])

runs = []
n_classes = None
for fname, label in RUNS:
    df, n_classes = load(FIGURE_DATA / fname)
    runs.append((label, df))

print(f"gene KO: {n_classes} classes | training sizes: {[l for l, _ in RUNS]}")

## Figure 4C

Single panel: viridis curves by training-set size, log x-axis (labels at 10 / 200 / 5000 with log minor ticks), percentage y-axis (majors every 20%, minors every 10%). Saves an SVG (paper) + PNG.

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 7.2))

for (label, df), color in zip(runs, COLORS):
    ax.plot(df["n_cells"].to_numpy(dtype=float), df["acc"].to_numpy(dtype=float),
            "-o", color=color, linewidth=6, markersize=19, zorder=3)

# x-axis: log, sparse labels + exponential (log) minor ticks
ax.set_xscale("log")
ax.set_xticks([10, 200, 5000])
ax.set_xticklabels(["10", "200", "5000"])
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10)))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("# cells per bag", fontsize=42)

# y-axis: 0-100%, majors every 20%, minors every 10%
ax.set_ylabel("Classification accuracy", fontsize=42)
ax.set_ylim(0.0, 1.0)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])
ax.yaxis.set_minor_locator(MultipleLocator(0.1))

ax.tick_params(axis="both", which="major", labelsize=36, width=2.5, length=13)
ax.tick_params(axis="both", which="minor", width=2, length=7)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
for sp in ("left", "bottom"):
    ax.spines[sp].set_linewidth(2)

# legend: largest training pool on top; line+marker handles (no error caps)
handles = [Line2D([0], [0], color=c, marker="o", markersize=17, linewidth=6, linestyle="-")
           for (_, lbl), c in zip(RUNS, COLORS)]
labels = [lbl for _, lbl in RUNS]
leg = ax.legend(handles[::-1], labels[::-1], title="Training set size",
                fontsize=28, title_fontsize=28, frameon=False,
                loc="upper left", handlelength=1.4)
leg._legend_box.align = "left"

fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_data_scaling.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "training_data_scaling.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Top-1 accuracy per (training set, n_cells), the values plotted above.

In [ ]:
rows = []
for label, df in runs:
    for _, r in df.iterrows():
        rows.append({"training_set": label, "n_cells": int(r["n_cells"]),
                     "top1_acc": float(r["acc"])})

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "training_data_scaling_summary.csv", index=False)
summary